<a href="https://colab.research.google.com/github/beagrs/Fundamental-of-Inference-and-Learning/blob/main/cifar100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 3: optimization of a CNN model
The task of this homework is to optimize a CNN model for the CIFAR-100. You are free to define the architecture of the model, and the training procedure. The only contraints are:
- It must be a `torch.nn.Module` object
- The number of trained parameters must be less than 1 million
- The test dataset must not be used for any step of training. It is better if don't even import it.
- The final training notebook should run on Google Colab within a maximum 1 hour approximately.

For the grading, you must use the `evaluate` function defined below. It takes a model as input, and returns the test accuracy as output.

As a guideline, you are expected to **discuss** and motivate your choices regarding:
- Model architecture
- Hyperparameters (learning rate, batch size, etc)
- Regularization methods
- Optimizer
- Validation scheme

A code without any explanation of the choices will not be accepted. Test accuracy is not the only measure of success for this homework.

Remember that most of the train process is randomized, store your model's weights after training and load it before the evaluation!

## Example

### Loading packages and libraries

In [ ]:
import torch
import torchvision
#from evaluate import evaluate

# Import the best device available
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
print('Using device:', device)

# load the data
train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=torchvision.transforms.ToTensor())

Using device: cpu


100%|██████████| 169M/169M [00:02<00:00, 65.5MB/s]


Extracting ./data/cifar-100-python.tar.gz to ./data


### Example of a simple CNN model

In [ ]:
class TinyNet(torch.nn.Module):
    def __init__(self):
        super(TinyNet, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = torch.nn.Linear(8*8*64, 128)
        self.fc2 = torch.nn.Linear(128, 100)

    def forward(self, x):
        x = torch.nn.functional.relu(self.conv1(x))
        x = torch.nn.functional.max_pool2d(x, 2)
        x = torch.nn.functional.relu(self.conv2(x))
        x = torch.nn.functional.max_pool2d(x, 2)
        x = x.view(-1, 8*8*64)
        x = torch.nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return x

print("Model parameters: ", sum(p.numel() for p in TinyNet().parameters()))

Model parameters:  556708


### Example of basic training

In [ ]:

model = TinyNet()
model.to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters())

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
for epoch in range(10):
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print('Epoch [{}/{}], Loss: {:.4f}'.format(epoch+1, 10, loss.item()))


Epoch [1/10], Loss: 4.6076
Epoch [2/10], Loss: 4.6036
Epoch [3/10], Loss: 4.5820
Epoch [4/10], Loss: 4.5764
Epoch [5/10], Loss: 4.5538
Epoch [6/10], Loss: 4.5622
Epoch [7/10], Loss: 4.4829
Epoch [8/10], Loss: 4.4569
Epoch [9/10], Loss: 4.3760
Epoch [10/10], Loss: 4.4282


In [ ]:
# save the model on a file
torch.save(model.state_dict(), 'tiny_net.pt')

loaded_model = TinyNet()
loaded_model.load_state_dict(torch.load('tiny_net.pt', weights_only=True))


<All keys matched successfully>

In [ ]:
from typing import Optional
from torch import nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torch.nn.functional as F
import numpy as np
from training_utils import *

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

Using device: cpu


The first step is to transform the dataset in tensor (requirement for feeding data into the model), then we split the dataset into train and test (i choose to keep a proportion of 80% for the training and 20% for the test)

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms

# Define the batch size
BATCH_SIZE = 128

# Define transformation pipeline: Convert images to tensors
transform_pipeline = transforms.ToTensor()

# Load the CIFAR-100 dataset with the transform applied
dataset = torchvision.datasets.CIFAR100(
    root='./data/',
    train=True,
    download=True,
    transform=transform_pipeline
)

# Set random seed for reproducibility
torch.manual_seed(42)

# Dynamically split the dataset into training and validation sets (80-20 split)
train_size = int(0.8 * len(dataset))
valid_size = len(dataset) - train_size
train_data, valid_data = random_split(dataset, [train_size, valid_size])

# Create DataLoader for training data
train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

# Create DataLoader for validation data
valid_loader = DataLoader(
    valid_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print(f'Training samples: {len(train_data)}')
print(f'Validation samples: {len(valid_data)}')


Files already downloaded and verified
Training samples: 40000
Validation samples: 10000


I exploit the function form the laboratory T

In [ ]:
from typing import Optional

def fit(
    model: nn.Module,
    train_dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    epochs: int,
    device: torch.device,
    val_dataloader: Optional[DataLoader] = None,
    scheduler_lr: Optional[torch.optim.lr_scheduler._LRScheduler] = None
):
    """
    the fit method simply calls the train_epoch() method for a
    specified number of epochs.
    """

    # keep track of the losses in order to visualize them later
    train_losses = []
    val_losses = []
    val_accuracies = []
    for epoch in range(epochs):
        train_loss = train_epoch(
            model=model,
            train_dataloader=train_dataloader,
            optimizer=optimizer,
            device=device,
        )
        train_losses.append(train_loss)
        if val_dataloader is not None:
            val_loss, val_accuracy = predict(
                model=model, test_dataloader=val_dataloader, device=device, verbose=False
            )
            val_losses.append(val_loss)
            val_accuracies.append(val_accuracy)
            print(
                f"Epoch {epoch}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.0f}%"
            )
        else:
            print(f"Epoch {epoch}: Train Loss={train_loss:.4f}")
        if scheduler_lr is not None:
            scheduler_lr.step()

    return train_losses, val_losses, val_accuracies

At this point, i start building the network .

The residual block employed, use a residual connection that allows the model to learn identity mappings if they are optimal. This helps mitigate the vanishing gradient problem in deep networks, facilitating better gradient flow during backpropagation. The residual connection is implemented by adding the output of the block to the input (out += self.skip(x)), which allows gradients to bypass certain layers if necessary, making training more stable. After each convolution operation, batch normalization is applied, which normalizes the activations within each mini-batch. This reduces internal covariate shift, stabilizing training and speeding up convergence. Additionally, dropout is included after the second convolution in each residual block. This regularizes the model by randomly setting a fraction of the activations to zero during training, helping to prevent overfitting by reducing reliance on any single feature or neuron.

The skip connection is handled with a 1x1 convolution if the dimensions of the input and output differ, ensuring that both the original and transformed inputs match in size before they are added together. This ensures that the residual connection operates smoothly even when the number of filters or the spatial resolution changes.

The overall network architecture is designed with multiple layers of residual blocks. The first layer of the network consists of a 3x3 convolution with 16 filters, which is a common starting point in CNN architectures due to its ability to efficiently capture local features. The network is divided into three residual layers (layer1, layer2, layer3), with the number of filters increasing from 16 to 64 as the network deepens. The increasing number of filters allows the model to capture more complex features at higher levels of abstraction. In the second and third layers, the stride is set to 2, reducing the spatial dimensions of the input and focusing the network on capturing more abstract features while reducing computational cost.

After passing through the residual blocks, global average pooling is applied. This operation reduces the spatial dimensions of the output to a fixed size of 1x1 for each feature map, making the architecture invariant to the input size and ensuring that the model can handle images of different resolutions. The output from the pooling layer is then flattened into a 1D vector before being passed to a fully connected layer for final classification. The number of output units in the fully connected layer corresponds to the number of classes in the dataset (default is 100), which makes the model suitable for multi-class classification tasks.

For weight initialization, the convolutional and linear layers are initialized using Kaiming normal initialization. This method is specifically designed for layers that use the ReLU activation function, helping to maintain variance across layers and speeding up convergence during training. Batch normalization layers are initialized with a weight of 1 and a bias of 0, which is a standard approach for these layers.

Finally, the model checks the total number of trainable parameters and ensures that it does not exceed 1 million,


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Define the Residual Block with Dropout and Batch Normalization
class ResidualBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1, dropout_prob=0.3):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        # Dropout for regularization
        self.dropout = nn.Dropout(p=dropout_prob)

        # Skip connection
        self.skip = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.skip = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.dropout(out)

        out += self.skip(x)  # Adding the skip connection
        out = self.relu(out)
        return out

In [ ]:
class HybridNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=100):
        super(HybridNet, self).__init__()
        self.in_planes = 16  # Initial number of filters

        # First convolution layer
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # Residual Layers (with increasing number of filters)
        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)

        # Global Average Pooling and Fully Connected Layer
        self.fc = nn.Linear(64, num_classes)

        # Initialize weights
        self._initialize_weights()

    def _make_layer(self, block, planes, num_blocks, stride):
        layers = []
        for i in range(num_blocks):
            layers.append(block(self.in_planes, planes, stride if i == 0 else 1))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                torch.nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                torch.nn.init.constant_(m.weight, 1)
                torch.nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                torch.nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                torch.nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        # Global Average Pooling
        x = F.adaptive_avg_pool2d(x, (1, 1))  # Ensures output size is (1, 1)
        x = x.view(x.size(0), -1)  # Flatten the output

        # Fully connected layer for classification
        x = self.fc(x)
        return x


In [ ]:
model = HybridNet(ResidualBlock, [2, 2, 2]).to(DEVICE)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params}")
assert num_params < 1_000_000

Model parameters: 181108


To choose the optimizer, i took into account two options: Adam or SGD with momenutm.
Adam is generally the go-to optimizer if we are working with deep networks or complex architectures and want to avoid spending too much time on hyperparameter tuning. It is highly effective in settings where training time and efficiency are important.

SGD with momentum is a solid choice if you need more control over the optimization process and are willing to fine-tune the parameters. It might be better suited for simpler models or when working with well-regularized data where you're looking for potentially better generalization, which is our case so i decide to go with SGD with momentum.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)


For the scheduler, as we have a small datasets and we whenwant to prevent overfitting, ReduceLROnPlateau is a good option because it dynamically adjusts the learning rate based on validation performance.

In [ ]:
scheduler_lr = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=4)

In [ ]:
train_losses, valid_losses, valid_accs =   fit(
        model,
        train_dataloader = train_loader,
        optimizer = optimizer,
        epochs = 30,
        device = DEVICE,
        val_dataloader = valid_loader,
        scheduler_lr = scheduler_lr
    )

plot_loss(train_losses)

At this stage, i perform data augementation and see if i ipmorove the performance of hte network

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Define data augmentation for training
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.75, 1.0)),  # Random crop with resizing
    transforms.RandomHorizontalFlip(p=0.5),  # Flip images horizontally with a 50% chance
    transforms.RandomVerticalFlip(p=0.2),    # Flip images vertically with a 20% chance
    transforms.RandomRotation(20),          # Rotate images within ±20 degrees
    transforms.ColorJitter(
        brightness=0.3, contrast=0.3, saturation=0.3, hue=0.2
    ),  # Color jitter for brightness, contrast, etc.
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])  # CIFAR-100 stats
])

# Define data transformation for validation (CIFAR-100 specific)
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5071, 0.4867, 0.4408], std=[0.2675, 0.2565, 0.2761])
])

# Load the CIFAR-100 dataset with the transform applied
dataset_aug = torchvision.datasets.CIFAR100(
    root='./data/',
    train=True,
    download=True,
    transform=train_transform
)

dataset_validation  = torchvision.datasets.CIFAR100(
    root='./data/',
    train=False,
    download=True,
    transform=val_transform
)
# Set random seed for reproducibility
torch.manual_seed(42)

# Dynamically split the dataset into training and validation sets (80-20 split)
train_size_aug = int(0.8 * len(dataset_aug))
valid_size_aug = len(dataset_aug - train_size)
train_data_aug, valid_data_aug = random_split(dataset_aug, [train_size_aug, valid_size_aug])

# Create DataLoader for training data
train_loader = DataLoader(
    train_data_aug,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

# Create DataLoader for validation data
valid_loader = DataLoader(
    dataset_validation,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)





Files already downloaded and verified
Files already downloaded and verified


In [ ]:
model = HybridNet(ResidualBlock, [2, 2, 2]).to(DEVICE)

# Count parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params}")
assert num_params < 1_000_000
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
scheduler_lr = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=4)

train_losses, valid_losses, valid_accs =   fit(
        model,
        train_dataloader = train_loader,
        optimizer = optimizer,
        epochs = 30,
        device = DEVICE,
        val_dataloader = valid_loader,
        scheduler_lr = scheduler_lr
    )

plot_loss(train_losses)

In [ ]:
torch.save(model.state_dict(), 'CNN_aug.pt')